In [1]:
import plotly.express as px

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os 
pop_info_file = "albo_global_pops_UNSD.txt"

pop_table = pd.read_table(pop_info_file,sep='\t', header=None )
pop_table.index = pop_table[0]
pop_table.columns = ['indv','pop','native_invasive']


# plink does weird thing to the ids
simplify_label = lambda x : x[ :len( x )//2]

In [2]:
def load_pca( baseFolder , prefix , add_native_invasive = True ):

    eig_val = pd.read_table( os.path.join( baseFolder , prefix + '.eigenval' ) , header=None )[0]
    #eig_val
    eig_vec = pd.read_table( os.path.join( baseFolder , prefix + '.eigenvec' ) , header=None , sep = ' ' , index_col=0)
    eig_vec.drop(columns=1 , inplace=True)
    eig_vec.columns = [ str(i) for i in range(eig_vec.shape[1]) ]
    eig_vec.index = [simplify_label(x) for x in eig_vec.index]
    eig_vec['pop'] = pop_table.loc[eig_vec.index , 'pop'].to_list()
    eig_vec['native_invasive'] = pop_table.loc[eig_vec.index , 'native_invasive'].to_list()
    
    if add_native_invasive: # add the native / invacive status to the name
        eig_vec.index = eig_vec['native_invasive'] + '@' + eig_vec.index
    
    return eig_val , eig_vec



In [8]:
## all individuals 
eig_val , eig_vec = load_pca( "." , "092_PCA_no_indonesia/FM_0.70_mD_2_MD_30_FMi_0.35_LD_thin.siblings_removed.no_indonesia" )
# plt.plot(  eig_val[:50] / eig_val.sum() )
# plt.xlabel("PC")
# plt.ylabel("var prop")
print( "PC0:{:.3f} , PC1:{:.3f} , PC2:{:.3f}".format( eig_val[0] / eig_val.sum() , eig_val[2] / eig_val.sum() , eig_val[3] / eig_val.sum() ) )
fig = px.scatter_3d( eig_vec , x = '0' , y = '3' , z='4' ,
                symbol = 'native_invasive' , color = 'pop' )

fig.write_html( '092_PCA_no_indonesia/FM_0.70_mD_2_MD_30_FMi_0.35_LD_thin.siblings_removed.no_indonesia.html' )
fig

PC0:0.017 , PC1:0.010 , PC2:0.010


In [11]:
eig_vec

,0,1,2,3,4,5,6,7,8,9,...,292,293,294,295,296,297,298,299,pop,native_invasive
invaded@ALKR1,0.030656,0.004606,0.007499,0.003687,-0.000177,0.006308,0.014301,-0.000078,-0.001903,-0.006265,...,-0.010868,0.005693,0.019228,-0.061819,0.005979,0.026789,-0.030044,0.068375,Albania,invaded
invaded@ALKR3,0.049063,0.012816,0.005836,0.010646,-0.005682,0.005576,0.018536,-0.006993,-0.002041,-0.017226,...,0.069940,-0.123433,0.067873,0.036176,0.058438,-0.011398,-0.095101,-0.010610,Albania,invaded
invaded@ALKR5,0.044160,0.016584,0.007542,0.007551,-0.005540,0.010294,0.028673,-0.007670,-0.005322,-0.007771,...,-0.055888,0.175576,-0.076176,0.134773,-0.055638,0.030914,-0.062781,-0.000394,Albania,invaded
invaded@ALSH2,0.031728,0.005558,0.003151,0.008033,-0.003754,0.001738,0.008059,-0.007473,-0.000842,-0.011879,...,0.217050,-0.251501,0.096009,0.128321,-0.029614,0.028349,-0.029444,-0.029404,Albania,invaded
invaded@ALTI3,0.044206,0.011939,0.000554,0.008907,0.001384,0.002189,0.036455,-0.004804,-0.005618,0.003539,...,0.054603,-0.051199,-0.010471,-0.042184,0.109998,0.011950,-0.022104,0.078625,Albania,invaded
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
native@vt18_10,-0.022309,-0.022341,-0.007736,-0.007677,0.002359,-0.003627,-0.058137,0.014278,0.007342,0.014821,...,-0.016827,0.009676,-0.001516,0.006617,-0.014632,0.013523,-0.002479,0.013918,Vietnam,native
native@vt18_11,-0.025886,-0.017890,-0.000348,0.004096,0.013346,0.004284,-0.035851,-0.007202,0.011271,0.002765,...,-0.008305,-0.010642,-0.003925,0.005002,0.001072,0.004741,0.013043,0.004367,Vietnam,native
native@vt18_12,-0.031748,-0.025525,-0.002194,-0.006878,0.000599,-0.002103,-0.044183,-0.006154,0.002982,0.020918,...,-0.005919,0.003574,0.007726,-0.005692,0.004492,-0.005400,-0.002822,-0.011924,Vietnam,native
native@vt18_13,-0.042741,-0.029501,0.015141,-0.017531,0.012884,0.008671,-0.058931,0.004731,0.021181,0.015081,...,0.019732,0.000197,-0.014024,-0.003939,0.002144,0.000527,-0.014046,0.004186,Vietnam,native


In [25]:
import umap

red = umap.UMAP(n_components=2)
red.fit( eig_vec.iloc[ : , :300 ] )

UMAP(tqdm_kwds={'bar_format': '{desc}: {percentage:3.0f}%| {bar} {n_fmt}/{total_fmt} [{elapsed}]', 'desc': 'Epochs completed', 'disable': True})

In [26]:
tmp = pd.DataFrame( red.embedding_ )
tmp.columns = ['x','y']
tmp.index= eig_vec.index
tmp['pop'] = eig_vec['pop']
tmp['native_invasive'] = eig_vec['native_invasive']
tmp

,x,y,pop,native_invasive
invaded@ALKR1,-0.382794,7.655054,Albania,invaded
invaded@ALKR3,-0.371172,7.675046,Albania,invaded
invaded@ALKR5,-0.836454,8.537564,Albania,invaded
invaded@ALSH2,-0.598375,8.457817,Albania,invaded
invaded@ALTI3,-0.260100,9.231972,Albania,invaded
...,...,...,...,...
native@vt18_10,-0.976058,10.684832,Vietnam,native
native@vt18_11,-2.177890,9.454452,Vietnam,native
native@vt18_12,-1.999624,10.275656,Vietnam,native
native@vt18_13,-2.006696,10.020579,Vietnam,native


In [27]:
px.scatter( tmp, x = 'x' , y = 'y' , color = 'pop' , symbol = 'native_invasive' )

In [7]:
## all individuals 
eig_val , eig_vec = load_pca( "." , "092_PCA_no_indonesia/FM_0.70_mD_2_MD_30_FMi_0.35_LD_thin.siblings_removed.no_indonesia.native" )
# plt.plot(  eig_val[:50] / eig_val.sum() )
# plt.xlabel("PC")
# plt.ylabel("var prop")
print( "PC0:{:.3f} , PC1:{:.3f} , PC2:{:.3f}".format( eig_val[0] / eig_val.sum() , eig_val[1] / eig_val.sum() , eig_val[2] / eig_val.sum() ) )
fig = px.scatter_3d( eig_vec , x = '0' , y = '1' , z='2' ,
                symbol = 'native_invasive' , color = 'pop' )

fig.write_html( '092_PCA_no_indonesia/FM_0.70_mD_2_MD_30_FMi_0.35_LD_thin.siblings_removed.no_indonesia.native.html' )
fig

PC0:0.034 , PC1:0.023 , PC2:0.022
